In [4]:
%pip install -U huggingface_hub

DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/dill-0.3.9-py3.12.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/looseversion-1.3.0-py3.12.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/opt_einsum-3.4.0-py3.12.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/nvfuser-0.2.25a0+6627725-py3.12-linux-x86_64.egg is deprecated. pip 25.1 wi

In [5]:
from huggingface_hub import snapshot_download

model_path = snapshot_download(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    local_dir="models/Qwen2.5-7B-Instruct"
)

print("Model downloaded to:", model_path)

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 14 files: 100%|██████████| 14/14 [00:00<00:00, 320.14it/s]

Model downloaded to: /workspace/rehancot/COT WORK/models/Qwen2.5-7B-Instruct


In [6]:
%pip install -U transformers accelerate safetensors

DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/dill-0.3.9-py3.12.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/looseversion-1.3.0-py3.12.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/opt_einsum-3.4.0-py3.12.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/nvfuser-0.2.25a0+6627725-py3.12-linux-x86_64.egg is deprecated. pip 25.1 wi

In [7]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_path = "models/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_path)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype="auto",
    device_map="auto"
)

print("Model loaded successfully!")

Loading weights: 100%|██████████| 339/339 [00:05<00:00, 67.01it/s]


Model loaded successfully!


In [8]:
messages = [
    {
        "role": "user",
        "content": "Solve 23 × 17. Explain the steps clearly."
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True
).to(model.device)

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=1000,
        do_sample=False
    )

answer = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True
)

print(answer)

Certainly! To solve \(23 \times 17\), we can use the standard multiplication method, which involves breaking down the problem into simpler parts. Here’s how you can do it step-by-step:

### Step 1: Break Down the Multiplication
We can break down the multiplication of 23 and 17 into smaller, more manageable parts using the distributive property of multiplication over addition. This means we can express 23 as \(20 + 3\) and 17 as \(10 + 7\).

So, \(23 \times 17 = (20 + 3) \times (10 + 7)\).

### Step 2: Apply Distributive Property
Now, distribute each term in the first parenthesis to each term in the second parenthesis:
\[ (20 + 3) \times (10 + 7) = 20 \times 10 + 20 \times 7 + 3 \times 10 + 3 \times 7 \]

### Step 3: Calculate Each Term
Let's calculate each of these terms individually:
- \(20 \times 10 = 200\)
- \(20 \times 7 = 140\)
- \(3 \times 10 = 30\)
- \(3 \times 7 = 21\)

### Step 4: Add the Results
Now, add all these results together:
\[ 200 + 140 + 30 + 21 = 391 \]

### Final A

In [10]:

import json
import os
import re
import time
import torch

# --------------------------------------------------
# 1. CONFIGURATION
# --------------------------------------------------

JSON_FILE_PATH = "datasets/aqua dataset/aqua_train_converted.json"

OUTPUT_FILE_PATH = (
    "rehancot/reasoning dataset/next_50_cot_results.json"
)

INITIAL_MAX_TOKENS = 2048
RETRY_MAX_TOKENS = 4096
MAX_ATTEMPTS = 2

# --------------------------------------------------
# 2. LOAD DATASET
# --------------------------------------------------

with open(JSON_FILE_PATH, "r", encoding="utf-8") as file:
    dataset = json.load(file)

if isinstance(dataset, dict):
    for key in ["data", "train", "examples", "questions"]:
        if key in dataset and isinstance(dataset[key], list):
            dataset = dataset[key]
            break

if not isinstance(dataset, list):
    raise ValueError("Expected a JSON list of questions.")

first_10 = dataset[10:60]

if not first_10:
    raise ValueError("Dataset contains no questions.")

os.makedirs(os.path.dirname(OUTPUT_FILE_PATH), exist_ok=True)

print(f"Loaded {len(first_10)} questions.")
print("Model:", getattr(model.config, "_name_or_path", "Unknown"))

# --------------------------------------------------
# 3. QUESTION AND OPTIONS
# --------------------------------------------------

def get_question_text(item):
    question = item["question"]
    options = item["options"]

    if isinstance(options, list):
        options_text = "\n".join(
            f"{chr(65 + i)}. {option}"
            for i, option in enumerate(options)
        )
    else:
        options_text = str(options)

    return question, options_text


# --------------------------------------------------
# 4. PARSE REASONING AND FINAL ANSWER
# --------------------------------------------------

def parse_response(generated_ids):
    answer_text = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()

    match = re.search(
        r"Final\s+Answer\s*:\s*(.+)",
        answer_text,
        flags=re.IGNORECASE | re.DOTALL
    )

    if not match:
        return answer_text, "", False

    final_answer = match.group(1).strip()

    if not final_answer:
        return answer_text, "", False

    # Reject obviously unfinished answers.
    if final_answer.endswith(
        (":", "-", "because", "therefore", "and")
    ):
        return answer_text[:match.start()].strip(), final_answer, False

    reasoning = answer_text[:match.start()].strip()

    return reasoning, final_answer, True


# --------------------------------------------------
# 5. EOS TOKEN DETECTION
# --------------------------------------------------

eos_ids = set()

for value in [
    tokenizer.eos_token_id,
    model.generation_config.eos_token_id
]:
    if isinstance(value, int):
        eos_ids.add(value)
    elif isinstance(value, (list, tuple)):
        eos_ids.update(value)

# --------------------------------------------------
# 6. RESUME EXISTING RESULTS
# --------------------------------------------------

if os.path.exists(OUTPUT_FILE_PATH):
    with open(OUTPUT_FILE_PATH, "r", encoding="utf-8") as file:
        results = json.load(file)

    if not isinstance(results, list):
        raise ValueError("Existing output JSON must contain a list.")
else:
    results = []

completed_ids = {
    str(row["id"])
    for row in results
    if row.get("generation_status") == "complete"
    and row.get("generated_answer")
    and row.get("reasoning")
}

print(f"Previously completed: {len(completed_ids)}")


def save_results():
    temp_path = OUTPUT_FILE_PATH + ".tmp"

    with open(temp_path, "w", encoding="utf-8") as file:
        json.dump(results, file, indent=2, ensure_ascii=False)

    os.replace(temp_path, OUTPUT_FILE_PATH)


# --------------------------------------------------
# 7. GENERATE REASONING AND MEASURE PERFORMANCE
# --------------------------------------------------

total_start = time.perf_counter()
total_generation_time = 0.0

for index, item in enumerate(first_10, start=11):

    question_id = str(item.get("id", index))

    if question_id in completed_ids:
        print(f"\nSkipping completed question {index}.")
        continue

    question, options_text = get_question_text(item)

    prompt = f"""Solve this multiple-choice question.

Question:
{question}

Options:
{options_text}

Instructions:
- Reason carefully and concisely.
- Explain the essential steps without repetition.
- Select the best answer from the provided options.
- Do not invent an option.
- Finish with exactly this format:

Final Answer: <option letter and answer text>
"""

    messages = [{"role": "user", "content": prompt}]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    print(f"\n{'=' * 60}")
    print(f"Question {index}/{len(first_10)}")
    print(question)

    best_result = None

    for attempt in range(1, MAX_ATTEMPTS + 1):

        token_limit = (
            INITIAL_MAX_TOKENS
            if attempt == 1
            else RETRY_MAX_TOKENS
        )

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        start = time.perf_counter()

        with torch.inference_mode():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=token_limit,
                do_sample=False,
                use_cache=True
            )

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        elapsed = time.perf_counter() - start

        generated_ids = output_ids[
            0, inputs["input_ids"].shape[1]:
        ]

        token_ids = generated_ids.tolist()
        token_count = len(token_ids)

        hit_limit = token_count >= token_limit
        ended_naturally = (
            bool(token_ids) and token_ids[-1] in eos_ids
        )

        reasoning, answer, answer_found = parse_response(
            generated_ids
        )

        is_complete = (
            answer_found
            and ended_naturally
            and not hit_limit
        )

        tokens_per_second = (
            token_count / elapsed if elapsed > 0 else 0
        )

        print(
            f"Attempt {attempt}: "
            f"{token_count} tokens, "
            f"{elapsed:.2f}s, "
            f"{tokens_per_second:.2f} tokens/sec, "
            f"limit reached={hit_limit}, "
            f"answer found={answer_found}, "
            f"natural stop={ended_naturally}"
        )

        current = {
            "id": item.get("id", index),
            "question": question,
            "options": options_text,
            "reasoning": reasoning,
            "generated_answer": answer,
            "generation_time_seconds": round(elapsed, 3),
            "generated_tokens": token_count,
            "tokens_per_second": round(tokens_per_second, 3),
            "attempts": attempt,
            "hit_token_limit": hit_limit,
            "ended_naturally": ended_naturally,
            "generation_status": (
                "complete" if is_complete else "incomplete"
            )
        }

        # Prefer a complete result. Otherwise keep the
        # attempt that generated the most tokens.
        if (
            best_result is None
            or (
                is_complete
                and best_result["generation_status"] != "complete"
            )
            or (
                not is_complete
                and best_result["generation_status"] != "complete"
                and token_count > best_result["generated_tokens"]
            )
        ):
            best_result = current

        if is_complete:
            print("Complete response accepted.")
            break

        if attempt < MAX_ATTEMPTS:
            print("Incomplete response; retrying with a larger limit.")

    total_generation_time += best_result["generation_time_seconds"]

    # Replace any old record with the same ID.
    results = [
        row for row in results
        if str(row.get("id")) != question_id
    ]

    results.append(best_result)
    save_results()

    if best_result["generation_status"] == "complete":
        completed_ids.add(question_id)

    print("\nREASONING:")
    print(best_result["reasoning"])

    print("\nFINAL ANSWER:")
    print(best_result["generated_answer"] or "[Missing]")

    print("STATUS:", best_result["generation_status"])
    print(f"Saved results to: {OUTPUT_FILE_PATH}")


# --------------------------------------------------
# 8. SUMMARY
# --------------------------------------------------

total_elapsed = time.perf_counter() - total_start

target_ids = {
    str(item.get("id", i))
    for i, item in enumerate(first_10, start=11)
}

current_results = [
    row for row in results
    if str(row.get("id")) in target_ids
]

complete_count = sum(
    row.get("generation_status") == "complete"
    for row in current_results
)

incomplete_count = len(current_results) - complete_count

total_tokens = sum(
    row.get("generated_tokens", 0)
    for row in current_results
)

average_tps = (
    total_tokens / total_generation_time
    if total_generation_time > 0
    else 0
)

print("\n" + "=" * 60)
print("GENERATION SUMMARY")
print("=" * 60)
print(f"Questions: {len(current_results)}")
print(f"Complete: {complete_count}")
print(f"Incomplete / needs review: {incomplete_count}")
print(f"Generated tokens: {total_tokens}")
print(f"Generation time this run: {total_generation_time:.2f}s")
print(f"Total elapsed time this run: {total_elapsed:.2f}s")
print(f"Overall generation throughput: {average_tps:.2f} tokens/sec")
print(f"Saved to: {OUTPUT_FILE_PATH}")


Loaded 50 questions.
Model: models/Qwen2.5-7B-Instruct
Previously completed: 0

Question 11/50
If Tim had lunch at $50 and he gave 20% tip, how much did he spend?
Attempt 1: 123 tokens, 1.25s, 98.29 tokens/sec, limit reached=False, answer found=True, natural stop=True
Complete response accepted.

REASONING:
To solve this problem, we need to calculate the total amount Tim spent, which includes both the cost of lunch and the tip.

1. The cost of lunch is $50.
2. The tip is 20% of $50. To find 20% of $50, we calculate \( 50 \times 0.20 = 10 \).
3. Therefore, the total amount spent is \( 50 + 10 = 60 \).

So, the correct answer is:

FINAL ANSWER:
A) $60.00
STATUS: complete
Saved results to: rehancot/reasoning dataset/next_50_cot_results.json

Question 12/50
Rs. 825 becomes Rs. 956 in 3 years at a certain rate of simple interest.If the rate of interest is increased by 4% ,What amount will Rs. 825 become in 3 years ?
Attempt 1: 468 tokens, 4.78s, 97.83 tokens/sec, limit reached=False, answer

In [11]:
import json
import re

JSON_FILE_PATH = "datasets/aqua dataset/aqua_train_converted.json"

OUTPUT_FILE_PATH = (
    "rehancot/reasoning dataset/next_50_cot_results.json"
)

with open(JSON_FILE_PATH, "r", encoding="utf-8") as f:
    dataset = json.load(f)

if isinstance(dataset, dict):
    for key in ["data", "train", "examples", "questions"]:
        if key in dataset and isinstance(dataset[key], list):
            dataset = dataset[key]
            break

with open(OUTPUT_FILE_PATH, "r", encoding="utf-8") as f:
    results = json.load(f)


def extract_option(value):
    """Extract a multiple-choice option letter when possible."""
    if value is None:
        return None

    text = str(value).strip().upper()

    # Handle formats such as "(C)", "C", or "C. answer".
    match = re.match(r"^\s*\(?([A-E])\)?(?:[\s.):\-]|$)", text)

    if match:
        return match.group(1)

    return None


# Map original records by their ID.
original_by_id = {
    str(item.get("id", i)): item
    for i, item in enumerate(dataset, start=1)
}

correct_count = 0
wrong_count = 0
unknown_count = 0

for result in results:
    original = original_by_id.get(str(result.get("id")))

    if original is None:
        print(f"ID {result.get('id')}: Original question not found")
        unknown_count += 1
        continue

    # Change "correct" if your dataset uses another answer key.
    correct = original.get("correct", original.get("answer"))
    generated = result.get("generated_answer", "")

    correct_letter = extract_option(correct)
    generated_letter = extract_option(generated)

    if correct_letter is None or generated_letter is None:
        print(f"\nQuestion ID: {result.get('id')}")
        print("Correct answer:", correct)
        print("Generated answer:", generated)
        print("Status: NEEDS MANUAL CHECK")
        unknown_count += 1
        continue

    is_correct = correct_letter == generated_letter

    if is_correct:
        correct_count += 1
    else:
        wrong_count += 1

    print(f"\nQuestion ID: {result.get('id')}")
    print("Correct option:", correct_letter)
    print("Generated option:", generated_letter)
    print("Status:", "CORRECT" if is_correct else "WRONG")

total_checked = correct_count + wrong_count

print("\n========== ACCURACY SUMMARY ==========")
print("Correct:", correct_count)
print("Wrong:", wrong_count)
print("Needs manual check:", unknown_count)

if total_checked:
    print(
        f"Answer accuracy: "
        f"{correct_count / total_checked * 100:.2f}%"
    )
else:
    print("No answers could be compared automatically.")


Question ID: 11
Correct option: A
Generated option: A
Status: CORRECT

Question ID: 12
Correct option: C
Generated option: B
Status: WRONG

Question ID: 13
Correct option: B
Generated option: B
Status: CORRECT

Question ID: 14
Correct option: D
Generated option: E
Status: WRONG

Question ID: 15
Correct option: C
Generated option: C
Status: CORRECT

Question ID: 16
Correct option: B
Generated option: B
Status: CORRECT

Question ID: 17
Correct option: D
Generated option: D
Status: CORRECT

Question ID: 18
Correct option: B
Generated option: B
Status: CORRECT

Question ID: 19
Correct option: A
Generated option: A
Status: CORRECT

Question ID: 20
Correct option: E
Generated option: E
Status: CORRECT

Question ID: 21
Correct option: D
Generated option: D
Status: CORRECT

Question ID: 22
Correct option: C
Generated option: C
Status: CORRECT

Question ID: 23
Correct option: C
Generated option: C
Status: CORRECT

Question ID: 24
Correct option: B
Generated option: B
Status: CORRECT

Question 

In [9]:

import json
import os
import re
import time
import torch

# ==================================================
# 1. CONFIGURATION
# ==================================================

JSON_FILE_PATH = "datasets/aqua dataset/aqua_train_converted.json"

OUTPUT_DIR = "rehancot/reasoning dataset"

CORRECT_FILE = os.path.join(
    OUTPUT_DIR, "aqua_cot_reasoning.json"
)

WRONG_FILE = os.path.join(
    OUTPUT_DIR, "aqua_cot_wrong_answers.json"
)

INITIAL_MAX_TOKENS = 2000
RETRY_MAX_TOKENS = 4000

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ==================================================
# 2. LOAD DATASET
# ==================================================

with open(JSON_FILE_PATH, "r", encoding="utf-8") as f:
    dataset = json.load(f)

if isinstance(dataset, dict):
    for key in ["data", "train", "examples", "questions"]:
        if isinstance(dataset.get(key), list):
            dataset = dataset[key]
            break

if not isinstance(dataset, list):
    raise ValueError("Dataset must contain a list of questions.")

if not dataset:
    raise ValueError("The dataset is empty.")

print("Total questions:", len(dataset))
print("Model:", getattr(model.config, "_name_or_path", "Unknown"))

# ==================================================
# 3. HELPERS FOR DATASET FIELDS
# ==================================================

def get_first(item, keys, default=None):
    for key in keys:
        if key in item and item[key] is not None:
            value = item[key]
            if value != "":
                return value
    return default


def get_options(item):
    options = get_first(item, ["options", "option", "choices"], [])

    if isinstance(options, dict):
        return options

    if isinstance(options, list):
        return options

    return str(options)


def format_options(options):
    if isinstance(options, dict):
        return "\n".join(
            f"{key}. {value}"
            for key, value in options.items()
        )

    if isinstance(options, list):
        return "\n".join(
            f"{chr(65 + i)}. {option}"
            for i, option in enumerate(options)
        )

    return str(options)


def get_correct_answer(item):
    # Adjust these names if your converted dataset uses
    # a different ground-truth answer field.
    return get_first(
        item,
        [
            "correct",
            "correct_answer",
            "correct_option",
            "answer",
            "label",
            "target"
        ]
    )


def get_concept(item):
    return get_first(
        item,
        ["concept", "topic", "category", "subject"],
        None
    )


def extract_option_letter(value):
    """Extract A-E from an answer such as 'C', '(C)', or 'C. 42'."""
    if value is None:
        return None

    text = str(value).strip().upper()

    match = re.match(
        r"^\s*\(?([A-E])\)?(?=\s|[.):\-]|$)",
        text
    )

    return match.group(1) if match else None


# ==================================================
# 4. PARSE THE MODEL RESPONSE
# ==================================================

def parse_response(generated_ids):
    text = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()

    match = re.search(
        r"Final\s+Answer\s*:\s*(.*)",
        text,
        flags=re.IGNORECASE | re.DOTALL
    )

    if not match:
        return text, "", False

    reasoning = text[:match.start()].strip()
    final_answer = match.group(1).strip()

    # Reject empty or obviously unfinished answers.
    if not final_answer:
        return reasoning, "", False

    if final_answer.endswith(
        (":", "-", "because", "therefore", "and")
    ):
        return reasoning, final_answer, False

    return reasoning, final_answer, True


# ==================================================
# 5. DETECT NATURAL GENERATION STOP
# ==================================================

eos_ids = set()

for value in [
    tokenizer.eos_token_id,
    model.generation_config.eos_token_id
]:
    if isinstance(value, int):
        eos_ids.add(value)
    elif isinstance(value, (list, tuple)):
        eos_ids.update(value)

# ==================================================
# 6. LOAD PREVIOUS RESULTS FOR RESUMING
# ==================================================

def load_results(path):
    if not os.path.exists(path):
        return []

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if not isinstance(data, list):
        raise ValueError(f"{path} must contain a JSON list.")

    return data


correct_results = load_results(CORRECT_FILE)
wrong_results = load_results(WRONG_FILE)

# A question is finished if it is already in either file.
finished_ids = {
    str(row["question_id"])
    for row in correct_results + wrong_results
    if "question_id" in row
}

print("Previously processed:", len(finished_ids))


def atomic_save(path, data):
    temp_path = path + ".tmp"

    with open(temp_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

    os.replace(temp_path, path)


# ==================================================
# 7. GENERATE REASONING FOR THE ENTIRE DATASET
# ==================================================

start_all = time.perf_counter()

correct_count = 0
wrong_count = 0
unverifiable_count = 0
skipped_count = 0

for index, item in enumerate(dataset, start=1):

    question_id = str(item.get("id", index))

    if question_id in finished_ids:
        skipped_count += 1
        continue

    question = get_first(item, ["question", "question_text"])

    if not question:
        print(f"Skipping ID {question_id}: missing question.")
        continue

    options = get_options(item)
    options_text = format_options(options)
    expected_answer = get_correct_answer(item)
    concept = get_concept(item)

    if expected_answer is None:
        raise ValueError(
            f"Missing ground-truth answer for question ID "
            f"{question_id}. Inspect the original JSON and update "
            f"get_correct_answer() before continuing."
        )

    expected_letter = extract_option_letter(expected_answer)

    if expected_letter is None:
        raise ValueError(
            f"Cannot extract an A-E answer letter for ID "
            f"{question_id}: {expected_answer!r}. "
            "Adapt answer normalization to your dataset format."
        )

    prompt = f"""Solve this multiple-choice question.

Question:
{question}

Options:
{options_text}

Instructions:
- Give concise, logically correct, step-by-step reasoning.
- Explain the essential calculations or deductions.
- Avoid repetition and unnecessary introductions.
- Select an answer from the given options only.
- Do not invent an option.
- Put the reasoning first.
- End with exactly this format on a new line:

Final Answer: <option letter and answer text>
"""

    messages = [{"role": "user", "content": prompt}]

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt"
    ).to(model.device)

    print(f"\n[{index}/{len(dataset)}] Question ID: {question_id}")
    print(question)

    accepted_result = None
    last_result = None

    # First 2000 tokens, then 4000 only if incomplete.
    for attempt, token_limit in enumerate(
        [INITIAL_MAX_TOKENS, RETRY_MAX_TOKENS],
        start=1
    ):

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        with torch.inference_mode():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=token_limit,
                do_sample=False,
                use_cache=True
            )

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        generated_ids = output_ids[
            0, inputs["input_ids"].shape[1]:
        ]

        token_ids = generated_ids.tolist()
        token_count = len(token_ids)

        hit_limit = token_count >= token_limit
        ended_naturally = (
            bool(token_ids) and token_ids[-1] in eos_ids
        )

        reasoning, final_answer, answer_found = parse_response(
            generated_ids
        )

        # Require a parsed answer and a natural stop.
        is_complete = (
            answer_found
            and ended_naturally
            and not hit_limit
        )

        last_result = {
            "reasoning": reasoning,
            "final_answer": final_answer,
            "answer_found": answer_found,
            "is_complete": is_complete
        }

        if is_complete:
            accepted_result = last_result
            break

        if attempt == 1:
            print("Incomplete response. Retrying with 4000 tokens.")

    # Use the final complete response if available; otherwise
    # keep the latest response for the review file.
    chosen = accepted_result if accepted_result else last_result

    reasoning = chosen["reasoning"]
    final_answer = chosen["final_answer"]
    generated_letter = extract_option_letter(final_answer)

    # --------------------------------------------------
    # 8. CHECK AGAINST GROUND-TRUTH ANSWER
    # --------------------------------------------------

    is_correct = (
        chosen["is_complete"]
        and generated_letter is not None
        and generated_letter == expected_letter
    )

    common_record = {
        "question_id": item.get("id", index),
        "question": question,
        "concept": concept,
        "option": options,
        "reasoning_generated": reasoning,
        "final_ans": final_answer
    }

    if is_correct:
        correct_results.append(common_record)
        atomic_save(CORRECT_FILE, correct_results)

        correct_count += 1
        print("ACCEPTED: correct answer.")
    else:
        # Incorrect or incomplete responses are excluded
        # from the main training dataset.
        wrong_record = {
            **common_record,
            "expected_ans": expected_answer,
            "generated_option": generated_letter,
            "generation_status": (
                "wrong_answer"
                if chosen["is_complete"] and generated_letter is not None
                else "incomplete_or_unparseable"
            )
        }

        wrong_results.append(wrong_record)
        atomic_save(WRONG_FILE, wrong_results)

        if chosen["is_complete"] and generated_letter is not None:
            wrong_count += 1
            print(
                f"REJECTED: expected {expected_letter}, "
                f"generated {generated_letter}."
            )
        else:
            unverifiable_count += 1
            print("REJECTED: incomplete or unparseable response.")

    finished_ids.add(question_id)

# ==================================================
# 9. FINAL SUMMARY
# ==================================================

elapsed_all = time.perf_counter() - start_all

print("\n" + "=" * 60)
print("FULL DATASET GENERATION SUMMARY")
print("=" * 60)
print("Total source questions:", len(dataset))
print("Accepted correct answers this run:", correct_count)
print("Rejected wrong answers this run:", wrong_count)
print("Incomplete/unparseable this run:", unverifiable_count)
print("Previously processed/skipped:", skipped_count)
print("Correct-results file:", CORRECT_FILE)
print("Wrong/review file:", WRONG_FILE)
print(f"Elapsed time this run: {elapsed_all / 60:.2f} minutes")
print("Finished.")


Total questions: 97467
Model: models/Qwen2.5-7B-Instruct
Previously processed: 0

[1/97467] Question ID: 1
Two friends plan to walk along a 43-km trail, starting at opposite ends of the trail at the same time. If Friend P's rate is 15% faster than Friend Q's, how many kilometers will Friend P have walked when they pass each other?
ACCEPTED: correct answer.

[2/97467] Question ID: 2
In the coordinate plane, points (x, 1) and (5, y) are on line k. If line k passes through the origin and has slope 1/5, then what are the values of x and y respectively?
ACCEPTED: correct answer.

[3/97467] Question ID: 3
For all numbers p and q, the operation @ is defined by p@q = p^2 - pq. If xy ≠ 0, then which of the following can be equal to zero?
I. x@y
II. (xy)@y
III. x@(x + y)
ACCEPTED: correct answer.

[4/97467] Question ID: 4
Carl is facing very difficult financial times and can only pay the interest on a $10,000 loan he has taken. The bank charges him a quarterly compound rate of 4%. What is the ap

KeyboardInterrupt: 